In [1]:
import os
import numpy as np
import torch
import cv2
from torch import nn
from pathlib import Path
from collections import defaultdict
from ultralytics import YOLO

In [3]:
PATH_WEIGHT = r'weights_sdu_v3\best_sdu_v3.pt'  #r'yolo11m.pt'

In [ ]:
# torch на готовой модели YOLO не работает, нужно разбираться, попробую как рекомендуется методами от ultralytics
# NOT IT TO DO FOR WORK
import torch

model = YOLO(r'weights_sdu_v3\best_sdu_v3.pt')


onnx_program = torch.onnx.export(
    model=model,
    args=(1, 3, 768, 1024),
    f='my_onnx_sdu.onnx',
    opset_version=17,
    input_names=['images'],
    output_names=['output'],
    dynamic_axes=None
)
if onnx_program:
    onnx_program.save('my_onnx_sdu.onnx')

YOLO

In [ ]:
# import torch
# import torch.nn as nn
# class ClampedYOLO(nn.Module):
#     def __init__(self, model):
#         super().__init__()
#         self.model = model
#     def forward(self, x):
#         outputs = self.model(x)
#         if isinstance(outputs, (list, tuple)):
#             clamped = [torch.clamp(out, 0.0, 1.0) for out in outputs]
#         else:
#             clamped = torch.clamp(outputs, 0.0, 1.0)
#         return clamped

In [24]:
model = YOLO(r'weights_sdu_v3\best_sdu_v3.pt')#.model.eval()
model.overrides['max'] = 100
model.model.eval()
# clamped_model = ClampedYOLO(model.model).eval()

for m in model.model.modules():
    if hasattr(m, 'export'):
        print(m.export)
        m.export =  True
        print(m.export)

model.model.eval()

dummy_inp = torch.randn(1, 3, 640, 640)

# export
torch.onnx.export(
    model.model,
    
    dummy_inp,
    r'weights_sdu_v3\best_sdu_v3_max.onnx',
    input_names=['images'],
    output_names = ['output'],
    opset_version=19,
    dynamic_axes=None,
    do_constant_folding=True,
    verbose=False
)

False
True


In [27]:
# 1
# ultralytics
PATH_WEIGHT = r'weights_sdu_v3\best_sdu_v3.pt'  #r'yolo11m.pt'  736.1024
model = YOLO(PATH_WEIGHT)
# model.model.model[-1].export = True
# source=r'E:\DB_SDU\ds_for_rknn'
model.export(format='onnx',
             imgsz=[640, 640],
             opset=12,
             half = False,
             int8 = False,
            #  training = True,
             dynamic = False,
             device='cpu',
             nms = True,
             simplify=True)

Ultralytics 8.3.168  Python-3.10.18 torch-2.7.1+cu126 CPU (Intel Xeon E5-2667 v2 3.30GHz)
Model summary (fused): 92 layers, 25,842,076 parameters, 0 gradients, 78.7 GFLOPs

PyTorch: starting from 'weights_sdu_v3\best_sdu_v3.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (49.6 MB)

ONNX: starting export with onnx 1.17.0 opset 12...
ONNX: slimming with onnxslim 0.1.65...
ONNX: export success  4.7s, saved as 'weights_sdu_v3\best_sdu_v3.onnx' (98.8 MB)

Export complete (10.4s)
Results saved to C:\TASK_DETECT_DRONE\CODES_DETECT_DRONE\YOLO+calc_distance\weights_sdu_v3
Predict:         yolo predict task=detect model=weights_sdu_v3\best_sdu_v3.onnx imgsz=640  
Validate:        yolo val task=detect model=weights_sdu_v3\best_sdu_v3.onnx imgsz=640 data=data_yaml_sdu_v3.yaml  
Visualize:       https://netron.app


'weights_sdu_v3\\best_sdu_v3.onnx'

In [28]:
import onnx
model = onnx.load(r"weights_sdu_v3\best_sdu_v3.onnx")
# model = onnx.load(r'weights_sdu_v3\yolo9out\yolo11m_9out.onnx')
onnx.checker.check_model(model)
print(onnx.helper.printable_graph(model.graph))

graph main_graph (
  %images[FLOAT, 1x3x640x640]
) initializers (
  %model.model.0.conv.weight[FLOAT, 48x3x3x3]
  %model.model.0.conv.bias[FLOAT, 48]
  %model.model.1.conv.weight[FLOAT, 96x48x3x3]
  %model.model.1.conv.bias[FLOAT, 96]
  %model.model.2.cv1.conv.weight[FLOAT, 96x96x1x1]
  %model.model.2.cv1.conv.bias[FLOAT, 96]
  %model.model.2.m.0.cv1.conv.weight[FLOAT, 48x48x3x3]
  %model.model.2.m.0.cv1.conv.bias[FLOAT, 48]
  %model.model.2.m.0.cv2.conv.weight[FLOAT, 48x48x3x3]
  %model.model.2.m.0.cv2.conv.bias[FLOAT, 48]
  %model.model.2.m.1.cv1.conv.weight[FLOAT, 48x48x3x3]
  %model.model.2.m.1.cv1.conv.bias[FLOAT, 48]
  %model.model.2.m.1.cv2.conv.weight[FLOAT, 48x48x3x3]
  %model.model.2.m.1.cv2.conv.bias[FLOAT, 48]
  %model.model.2.cv2.conv.weight[FLOAT, 96x192x1x1]
  %model.model.2.cv2.conv.bias[FLOAT, 96]
  %model.model.3.conv.weight[FLOAT, 192x96x3x3]
  %model.model.3.conv.bias[FLOAT, 192]
  %model.model.4.cv1.conv.weight[FLOAT, 192x192x1x1]
  %model.model.4.cv1.conv.bias[FLO

In [30]:
# узнать точные имена и формы входных узлов onnx model
import onnx
model_check = onnx.load(r'weights_sdu_v3\best_sdu_v3.onnx')
# model_check = onnx.load(r'weights_sdu_v3\yolo9out\yolo11m_9out.onnx')
for inp in model_check.graph.input:
    print(f"Input name: {inp.name}")
print(f"Input shape: {[dim.dim_value for dim in inp.type.tensor_type.shape.dim]}")

outputs = model_check.graph.output
for output in outputs:
    print(output.name, 'shape: ', output.type.tensor_type.shape)

Input name: images
Input shape: [1, 3, 640, 640]
output0 shape:  dim {
  dim_value: 1
}
dim {
  dim_value: 300
}
dim {
  dim_value: 6
}



In [ ]:
m = onnx.load(r'best_sdu_v3.onnx')
print(m.graph.input)
for out in m.graph.output:
    print(out.name, out)

Сырой тест на onnxruntime для  onnx формата модели, чтобы получить сырой выход и разобрать возможно ли изменить постобработку для rknn

In [15]:
def letterbox(im, new_shape=(640, 640), color=(114, 114, 114)):
    shape = im.shape[:2]  # current shape [height, width]
    if isinstance(new_shape, int):
        new_shape = (new_shape, new_shape)

    # Scale ratio (new / old)
    r = min(new_shape[0] / shape[0], new_shape[1] / shape[1])
    new_unpad = int(round(shape[1] * r)), int(round(shape[0] * r))
    dw, dh = new_shape[1] - new_unpad[0], new_shape[0] - new_unpad[1]  # wh padding

    dw /= 2  # divide padding into 2 sides
    dh /= 2

    if shape[::-1] != new_unpad:  # resize
        im = cv2.resize(im, new_unpad, interpolation=cv2.INTER_LINEAR)
    top, bottom = int(round(dh - 0.1)), int(round(dh + 0.1))
    left, right = int(round(dw - 0.1)), int(round(dw + 0.1))
    im = cv2.copyMakeBorder(im, top, bottom, left, right, cv2.BORDER_CONSTANT, value=color)
    
    return im
def leter_box(img, new_shape = (640, 640),
        auto = False,
        scale_fill = False,
        scaleup = True,
        center = True,
        stride = 32,
        pading_value = (114, 114, 114),
        interpolation = cv2.INTER_LINEAR): #(int value))


    shape = img.shape[:2]
    new_shape = new_shape

    if isinstance(new_shape, int):
        new_shape = (new_shape, new_shape)

    #scale ratio
    r = min(new_shape[0]/shape[0], new_shape[1]/shape[1])

    #scale up
    if not scaleup:
        r = min(r, 1.0)

    # compute padding
    ratio = r, r # width, height ration
    new_unpad = int(round(shape[1] * r)), int(round(shape[0] * r))
    dw, dh = new_shape[1] - new_unpad[0], new_shape[0] - new_unpad[1] # w, h

    if auto:
        dw, dh = np.mod(dw, stride), np.mod(dh, stride) # wh padding
    elif scale_fill: # strech
        dw, dh = 0.0, 0.0
        new_unpad = (new_shape[1], new_shape[0])
        ratio = new_shape[1]/shape[1], new_shape[0]/shape[0]  # width, height

    if center:
        dw /= 2
        dh /= 2
    if shape[::-1] != new_unpad:
        img = cv2.resize(img, new_unpad, interpolation=interpolation)
        if img.ndim == 2:
            img = img[..., None]

    top, bottom = int(round(dh - 0.1)) if center else 0, int(round(dh + 0.1))
    left, right = int(round(dw - 0.1)) if center else 0, int(round(dw + 0.1))

    h, w, c = img.shape
    if c == 3:
        img = cv2.copyMakeBorder(
            img, top, bottom, left, right, cv2.BORDER_CONSTANT, value=(pading_value)
        )
    else: # multispectral
        pad_img = np.full((h + top + bottom, w + left + right, c), fill_value=pading_value, dtype=img.dtype)
        pad_img[top: top + h, left: left + w] = img
        img = pad_img
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img

def letterbox(im, new_shape=(640, 640), color=(114, 114, 114)):
    shape = im.shape[:2]  # current shape [height, width]
    if isinstance(new_shape, int):
        new_shape = (new_shape, new_shape)

    # Scale ratio (new / old)
    r = min(new_shape[0] / shape[0], new_shape[1] / shape[1])
    new_unpad = int(round(shape[1] * r)), int(round(shape[0] * r))
    dw, dh = new_shape[1] - new_unpad[0], new_shape[0] - new_unpad[1]  # wh padding

    dw /= 2  # divide padding into 2 sides
    dh /= 2

    if shape[::-1] != new_unpad:  # resize
        im = cv2.resize(im, new_unpad, interpolation=cv2.INTER_LINEAR)
    top, bottom = int(round(dh - 0.1)), int(round(dh + 0.1))
    left, right = int(round(dw - 0.1)), int(round(dw + 0.1))
    im = cv2.copyMakeBorder(im, top, bottom, left, right, cv2.BORDER_CONSTANT, value=color)
    
    return im

In [31]:
import onnxruntime as ort

#Инициализация модели
session = ort.InferenceSession(r'weights_sdu_v3\best_sdu_v3.onnx')
# session = ort.InferenceSession(r'weights_sdu_v3\yolo9out\yolo11m_9out.onnx')
# Получаем имя входного тензора
input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name


# prepare image
img = cv2.imread(r'E:\DB_SDU\ds_for_rknn\animal_orig.jpg') # F:\RADXA\ds_for_rknn\DJIG0132_1_1880.jpg  weights_sdu_v3\0000069_00713_d_0000003.jpg
# img = cv2.resize(img,(1024, 1024))
# img = img.astype(np.float32) / 255.0
img = letterbox(img)
img = img.astype(np.float32) / 255.0
img = np.transpose(img, (2, 0, 1))
img = np.expand_dims(img, axis=0)
print(img.shape)

# inference
output = session.run([output_name], {input_name: img})[0]
print(output[0].min(), output[0].max())

print("output shape: ", output.shape)
print("Raw output ")
print(output[0, :20, :5])


(1, 3, 640, 640)
-0.027366638 299.8673
output shape:  (1, 300, 6)
Raw output 
[[     98.055       236.4      115.44      253.05     0.85438]
 [      160.9      274.62      185.75      296.35     0.84829]
 [     25.779      277.04      45.459      299.87     0.84802]
 [     194.79      271.26      213.75      293.98     0.83948]
 [     219.85      259.98      236.19      281.81     0.83819]
 [     99.928      257.96       119.5      277.17     0.81648]
 [     107.46      270.08      126.08      292.61     0.80948]
 [     6.2601      262.53      25.926       280.8     0.80549]
 [  -0.027367      255.16      11.885      271.69     0.67051]
 [          0           0           0           0           0]
 [          0           0           0           0           0]
 [          0           0           0           0           0]
 [          0           0           0           0           0]
 [          0           0           0           0           0]
 [          0           0           0   

In [21]:
len(output)

1

In [ ]:
pred = np.transpose(output, (0, 2, 1))
print(pred.shape)
# for p in pred[0]:
#     print(p)
#     break
conf_trech = 0.1
iou_tresh = 0.2
detections = []
boxes = pred[0, :, :4]  # xc,yc, w, h
print('Shape boxes: ', boxes.shape)
scores = pred[0, :, 4:] # все score это логиты вывода модели
print('Scores: ', scores.shape)
class_idx = np.argmax(scores, axis=1)
print('cls: ', class_idx, len(class_idx))  # индесы максимальные со всех скоров по строке
confidences = scores[np.arange(len(scores)), class_idx]


mask = confidences > conf_trech

# boxes_ = boxes[mask]
# confds = confidences[mask]

# class_idx = class_idx[mask]
print(scores.shape, )
#NMS
indices = cv2.dnn.NMSBoxes(boxes, confidences, conf_trech, iou_tresh)
for i in indices:
    detections.append([
        boxes[i],
        confidences[i],
        class_idx[i]
    ])

(1, 6, 300)
Shape boxes:  (6, 4)
Scores:  (6, 296)
cls:  [0 2 0 2 0 0] 6
(6, 296)


In [10]:
print(len(detections))
for coord, conf, cl in detections:
    print(coord, ' - ', conf, ' - ', cl)

10
[     106.73      244.75      17.499      16.397]  -  0.85621715  -  3
[     35.689      288.48      19.823      22.707]  -  0.8502771  -  3
[     173.33      285.45      24.995       21.43]  -  0.8462827  -  3
[     227.98      270.93       16.45       21.63]  -  0.84377587  -  3
[     204.36      282.66      19.093      22.436]  -  0.8381971  -  3
[      109.7      267.57      19.694      18.987]  -  0.820891  -  3
[     116.87       281.2       18.69      22.258]  -  0.81566393  -  3
[     16.028      271.66       19.68      18.102]  -  0.80653226  -  3
[     6.0411      263.41       12.17      16.339]  -  0.7020966  -  3
[     231.24      168.06      12.349      8.7203]  -  0.15233952  -  3


In [36]:
img_o = cv2.imread(r'E:\DB_SDU\ds_for_rknn\animal.jpg') # usb - F:\RADXA\ds_for_rknn\DJIG0132_1_1880.jpg  weights_sdu_v3\0000069_00713_d_0000003.jpg
# img_o = cv2.resize(img_o,(1024, 1024))
for coord, conf, cl in detections:
    # print(coord, ' - ', conf, ' - ', cl)
    xc, yc, w, h = coord
    x1 = int(xc - w / 2)
    y1 = int(yc - h / 2)
    x2 = int(xc + w / 2)
    y2 = int(yc + h / 2)
    conf = str(conf)
    cl = str(cl)
    cv2.rectangle(img_o, (x1, y1), (x2, y2), (0, 255, 0), 1)
    cv2.putText(img_o, f'cl:{cl}_cf:{conf}', (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
cv2.imshow('onnx det', img_o)
cv2.waitKey(0)
cv2.destroyAllWindows()    

### Test models yolo11m.pt before convert rknn

In [4]:
path_img = r'imgs\0000069_00713_d_0000003.jpg'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [5]:
img = cv2.imread(path_img)
img = np.expand_dims(img, axis=0)
img = img.transpose((0, 3, 1, 2))
img.shape

(1, 3, 765, 1360)

In [6]:
model_tst = YOLO(r'weights_sdu_v3\best_sdu_v3.pt', task='detect')

In [37]:
img = cv2.imread(path_img)
img = cv2.resize(img, (1024, 768))

In [56]:
import time
start_time = time.time()
result = model_tst(img, imgsz=640, conf=0.43, iou=0.2, device='cuda')
print(f"Time {(time.time() - start_time) * 1000}")
annotated_frame = result[0].show()



0: 480x640 32 persons, 1 car, 263.4ms
Speed: 5.7ms preprocess, 263.4ms inference, 7.7ms postprocess per image at shape (1, 3, 480, 640)
Time 290.0195121765137
